# OpenForge free notebook workflow

Prepared workflow, not an executed cloud experiment. Use Kaggle Free only; no paid
endpoints or quota/account rotation. Default execution performs a six-task dry run,
revalidates it, exports offline HTML and compares the source with itself. No model
runtime is installed and no weights downloaded. GPU/runtime compatibility remains
unverified. Source is pinned to an immutable reviewed development snapshot below.


In [ ]:
from pathlib import Path
import json, subprocess, sys
REVISION = "b89b8dfdfb2ac6f977b493fc775d37bea5f7b02a"
REPO = Path.cwd() / "openforge-source"
if REPO.exists():
    raise RuntimeError("Use a fresh workspace; existing source is never overwritten")
subprocess.run(["git", "clone", "https://github.com/Light-Dataforge/openforge-ai.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", "--detach", REVISION], cwd=REPO, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
if actual != REVISION:
    raise RuntimeError("Source revision mismatch")
sys.path.insert(0, str(REPO))
print("Pinned source:", actual)


In [ ]:
from core.multimodal.fixtures import generate_fixtures
from apps.baseline.runner import run_pilot
from apps.pilot_verification.runner import verify_pilot
from apps.pilot_report.renderer import build_report
from apps.pilot_compare.runner import compare_pilots
ROOT = Path.cwd() / "openforge-artifacts"
ROOT.mkdir(mode=0o700, exist_ok=False)
manifest = generate_fixtures(ROOT / "fixtures")
dry = run_pilot(manifest, ROOT / "dry-run", dry_run=True)
assert dry["answer_calls"] == 0 and dry["not_executed"] == 6
verified = verify_pilot(ROOT / "dry-run", manifest, ROOT / "dry-verification")
assert verified["counts"]["not_executed"] == 6
report = build_report(ROOT / "dry-run", manifest, ROOT / "dry-report.html")
comparison = compare_pilots(ROOT / "dry-run", ROOT / "dry-run", manifest,
                            ROOT / "dry-self-comparison.json")
assert comparison["transitions"]["not_executed"]["not_executed"] == 6
print(json.dumps({"dry_run": dry, "verification": verified}, indent=2))


## Optional runtime preparation

Stop here to keep this entirely model-free. Read the pinned requirements and model
adapter instructions below before installing anything. Select official CPU/CUDA
wheels suitable for the actual runtime/driver. This cell only prints instructions;
no installation, upgrade, GPU allocation or download is automatic. The assembled
optional package stack has not been exercised in Kaggle.


In [ ]:
print((REPO / "requirements-vlm.txt").read_text())
print((REPO / "docs/pilot-model-adapter.md").read_text())


## Optional live run and retained evidence

Only after manually preparing a reviewed free runtime. RUN_MODEL and public weight
downloads both default to false; CPU is default. No training is implemented. The
operator must supervise the session budget and save artifacts before it expires.
Model-load/answer failures remain visible. A dry-run-to-live comparison documents
execution states; it is not an accuracy improvement or a statistical result.


In [ ]:
RUN_MODEL = False
ALLOW_DOWNLOAD = False
DEVICE = "cpu"
if RUN_MODEL:
    live = run_pilot(manifest, ROOT / "live-run", allow_download=ALLOW_DOWNLOAD,
                     device=DEVICE, max_new_tokens=64)
    checked = verify_pilot(ROOT / "live-run", manifest, ROOT / "live-verification")
    build_report(ROOT / "live-run", manifest, ROOT / "live-report.html")
    compare_pilots(ROOT / "dry-run", ROOT / "live-run", manifest,
                   ROOT / "dry-to-live-comparison.json")
    print(json.dumps({"source": live, "checks": checked}, indent=2))
else:
    print("No live inference, model runtime or weights requested.")


## Save and review artifacts

Download openforge-artifacts through the notebook file browser. Raw pilot records
and HTML contain answers/task metadata and may expose private text; review before
sharing. Comparison JSON excludes raw answers but retains IDs/digests/change flags.
These narrow synthetic pixel checks test pipeline behavior, not general model
accuracy. Adapter declarations do not authenticate the source runtime. This notebook
ships without saved outputs or a claimed cloud execution.
